# Pre-Requisites

## Load 'employee.csv' into DataFrame


1. Sparator = "|"
2. Quote = "'" (Removes the single quote)

In [0]:
df = spark.read.csv(
  path = "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/employee.csv",
  header = True,
  inferSchema = True,
  sep = "|"
)
df.display()

In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/employee.csv",
    header=True,
    inferSchema=True,
    sep="|",
    quote="'"
)
df.display()

# Select Columns

1. select by default datatype is "dataframe"
2. select parameters - columns to display

In [0]:
df.select()

In [0]:
# df.select("id","name").display() - Do not select like this. Select by using column object.

from pyspark.sql.functions import col
df.select(col("id"), col("name").alias("full_name")).display() #alias to rename

# Filter Records

In [0]:
df.filter(col("gen")=="M").select(col("name")).display()

In [0]:
#df.filter((col("gen")=="M") & (col("company")=="Infosys")).select("name").display()
df.filter(col("gen")=="M").filter(col("company")=="Infosys").select("name").display()

In [0]:
#Return type of lower() is column so no need to give col() inside lower()

from pyspark.sql.functions import lower
df.filter(lower("company") =="cisco").select("name").display()

In [0]:
# Create a new column -> lit - literal

from pyspark.sql.functions import lit
df.withColumn("is_employeed", lit("True")).display()

In [0]:
# Create a column category - Value is based on a condition relating some other column
# Ctrl + Shift + F -> Format

from pyspark.sql.functions import when

df.withColumn(
    "exp_category",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience"),
).select("name", "exp", "exp_category").display()

In [0]:
from pyspark.sql.functions import when

df.withColumn(
    "gender",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Invalid Gender"),
).select("name", "gen", "gender").display()

In [0]:
# Modification of a column value without creating a new column
from pyspark.sql.functions import when

df.withColumn(
    "gen",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Invalid Gender"),
).select("name", "gen").display()

In [0]:
df2 = df.withColumn(
    "gen",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Invalid Gender"),
).select("name", "gen")

df2.groupBy("gen").count().display()

In [0]:
df3 = df.withColumn(
    "exp_category",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience"),
).select("name", "exp", "exp_category")

df3.groupBy("exp_category").count().display()

In [0]:
#withColumnRenamed - Rename a column
#sort("col_name") - Sort by column name

df2 = df.withColumn(
    "gen",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Invalid Gender"),
).select("name", "gen")

df2.groupBy("gen").count().sort("count").display() #Ascending
df2.groupBy("gen").count().sort("count", ascending=False).display() #Descending

In [0]:
df3 = df.withColumn(
    "exp_category",
    when(((col("exp") >= 10) & (col("gen") == "M") | (col("exp") >= 9) & (col("gen") == "F")), "Senior")
    .when(((col("exp") >= 5) & (col("gen") == "M") | (col("exp") >= 4) & (col("gen") == "F")), "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience")
).select("name", "exp", "gen", "exp_category")

df3.display()
df3.groupBy("exp_category").count().display()